In [12]:
import pandas as pd
import os

In [2]:
RAW_FILE = "../data/raw/2019-Oct.csv"
OUTPUT_FILE = "../data/interim/filtered_sample.csv"

os.makedirs("../data/interim", exist_ok=True)


In [3]:
USE_COLUMNS = [
    "user_id",
    "product_id",
    "event_type",
    "event_time",
    "category_code"
]

CHUNK_SIZE = 500_000   # safe for 8GB RAM
TARGET_ROWS = 300_000  # expanded dataset size (was 100_000)

In [4]:
filtered_chunks = []
current_rows = 0

print("Starting chunk-wise reading...")

for chunk in pd.read_csv(
    RAW_FILE,
    usecols=USE_COLUMNS,
    chunksize=CHUNK_SIZE
):
    # keep only useful events
    chunk = chunk[
        chunk["event_type"].isin(["view", "cart", "purchase"])
    ]

    filtered_chunks.append(chunk)
    current_rows += len(chunk)

    print(f"Collected rows so far: {current_rows}")

    # stop early once enough data is collected
    if current_rows >= TARGET_ROWS * 2:
        break


Starting chunk-wise reading...
Collected rows so far: 500000
Collected rows so far: 1000000


In [5]:
df = pd.concat(filtered_chunks, ignore_index=True)

print("Total rows before sampling:", len(df))

df = df.sample(n=TARGET_ROWS, random_state=42)

print("Total rows after sampling:", len(df))


Total rows before sampling: 1000000
Total rows after sampling: 300000


In [6]:
df.to_csv(OUTPUT_FILE, index=False)

print("Saved filtered dataset to:", OUTPUT_FILE)


Saved filtered dataset to: ../data/interim/filtered_sample.csv


In [7]:
print("\nSample rows:")
print(df.head())

print("\nEvent type distribution:")
print(df["event_type"].value_counts())

print("\nUnique users:", df["user_id"].nunique())
print("Unique products:", df["product_id"].nunique())



Sample rows:
                     event_time event_type  product_id  \
987231  2019-10-01 16:48:11 UTC       view     4802401   
79954   2019-10-01 04:07:06 UTC       view     1005064   
567130  2019-10-01 11:24:05 UTC       view     1004870   
500891  2019-10-01 10:23:33 UTC       view    15100216   
55399   2019-10-01 03:39:28 UTC       view     8800448   

                      category_code    user_id  
987231  electronics.audio.headphone  514990201  
79954        electronics.smartphone  518083733  
567130       electronics.smartphone  539581936  
500891                          NaN  517852471  
55399         electronics.telephone  519838641  

Event type distribution:
event_type
view        290603
purchase      5041
cart          4356
Name: count, dtype: int64

Unique users: 105834
Unique products: 41934


In [8]:
# Remove sparse users
df = df.groupby("user_id").filter(lambda x: len(x) >= 3)

# Remove sparse products
df = df.groupby("product_id").filter(lambda x: len(x) >= 3)

print("After sparsity filtering:")
print("Rows:", len(df))
print("Users:", df["user_id"].nunique())
print("Products:", df["product_id"].nunique())


After sparsity filtering:
Rows: 179509
Users: 35516
Products: 14011


In [9]:
df["category_code"] = df["category_code"].fillna("unknown")


In [10]:
event_weight = {
    "view": 1,
    "cart": 3,
    "purchase": 5
}

df["event_strength"] = df["event_type"].map(event_weight)


In [11]:
df.to_csv(
    "../data/interim/filtered_sample_final.csv",
    index=False
)

print("✅ Final dataset saved as filtered_sample_final.csv")


✅ Final dataset saved as filtered_sample_final.csv


In [13]:
print(df.head())
print(df["event_type"].value_counts())


                     event_time event_type  product_id  \
199676  2019-10-01 06:06:39 UTC       view    10900328   
408697  2019-10-01 09:04:19 UTC       view     1306569   
163280  2019-10-01 05:33:14 UTC       view     1004356   
6940    2019-10-01 02:31:49 UTC       view     1004750   
230672  2019-10-01 06:34:34 UTC       view    17300752   

                   category_code    user_id  event_strength  
199676  appliances.kitchen.mixer  519282157               1  
408697        computers.notebook  517726252               1  
163280    electronics.smartphone  551377651               1  
6940      electronics.smartphone  519403613               1  
230672                   unknown  514966122               1  
event_type
view        37642
cart          969
purchase      867
Name: count, dtype: int64
